In [1]:
import os, openml, tqdm, json

In [2]:
import numpy as np
import pandas as pd

In [3]:
from pathlib import Path
from openml.tasks import TaskType
from sklearn.preprocessing import LabelEncoder

In [4]:
benchmark = openml.study.get_suite('tabarena-v0.1')
tasks = openml.tasks.list_tasks(task_id=benchmark.tasks, output_format='dataframe')
tasks = tasks[tasks['ttid'] == TaskType.SUPERVISED_CLASSIFICATION]
tasks = tasks['did'].astype(int).tolist()

In [5]:
def load_tabarena(id):
    ds = openml.datasets.get_dataset(id, download_data=True)
    X, y, _, _ = ds.get_data(target=ds.default_target_attribute)
    return X, LabelEncoder().fit_transform(y)

In [6]:
for id in tqdm.tqdm(tasks):
    X, y = load_tabarena(id)
    data = np.column_stack([X, y])
    pd.DataFrame(data).to_csv(f'tabarena/{id}.csv', index=False)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 38/38 [00:14<00:00,  2.63it/s]


In [7]:
def load_split(id, split):
    blocks = []
    for prefix in ['N', 'C']:
        path = f'temp/{id}/{prefix}_{split}.npy'
        if os.path.exists(path):
            blocks.append(np.load(path, allow_pickle=True))
    X = np.concatenate(blocks, axis=1)
    y = np.load(f'temp/{id}/y_{split}.npy', allow_pickle=True)
    return X, y

In [8]:
def load_talent(id):
    
    Xs, ys = [], []
    for split in ['train', 'test', 'val']:
        try:
            X, y = load_split(id, split)
            Xs.append(X)
            ys.append(y)
        except:
            continue
     
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return X, LabelEncoder().fit_transform(y)

In [9]:
tasks = os.listdir('temp/')

In [10]:
for id in tqdm.tqdm(tasks):
    info = json.loads(Path(f'temp/{id}/info.json').read_text())
    if info['task_type'] in ['binclass', 'multiclass']:
        X, y = load_talent(id)
        data = np.column_stack([X, y])
        pd.DataFrame(data).to_csv(f'talent/{id}.csv', index=False)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:16<00:00, 17.81it/s]
